# Setup & Imports

In [1]:
# ==============================================================================
# 1. Setup & Imports
# ==============================================================================
import yaml
from copy import deepcopy
from pathlib import Path

CONFIG_FILE_NAME = "campaign_config.yaml"


def merge_config_layer(base, override):
    """Merge one config layer into another, appending parameter_sets."""
    result = deepcopy(base)

    for key, value in override.items():
        if key == "parameter_sets":
            existing = result.get(key, []) or []
            additions = deepcopy(value) if value else []
            result[key] = [*existing, *additions]
        elif isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = merge_config_layer(result[key], value)
        else:
            result[key] = deepcopy(value)

    return result


def resolve_simulation_config(campaign_config, sim_id):
    """Merge campaign defaults, mode defaults, and simulation overrides."""
    simulations = campaign_config.get("simulations", [])
    sim_entry = next((entry for entry in simulations if entry.get("id") == sim_id), None)
    if sim_entry is None:
        raise KeyError(f"Simulation '{sim_id}' was not found in the campaign config.")

    merged_config = {}
    defaults = campaign_config.get("simulation_defaults", {})
    mode_defaults = campaign_config.get("mode_defaults", {}).get(sim_entry.get("mode"), {})

    for layer in (defaults, mode_defaults, sim_entry):
        merged_config = merge_config_layer(merged_config, layer)

    return merged_config


# Resolve paths
# __file__ is not available in Jupyter notebooks; fall back to Path().resolve()
# which returns the notebook's working directory (its own folder when run in VS Code).
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path().resolve()

CAMPAIGN_DIR = SCRIPT_DIR
PARAMS_DIR = CAMPAIGN_DIR / "parameters"
OPT_CONFIG_DIR = CAMPAIGN_DIR / "optimization_config"

# ── AnimaKinetica imports ──────────────────────────────────────────────────────
from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.core.simulation_data import SimulationData

from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.perfusion_strategies import PerfusionStrategy
from animakinetica.models.gas_controllers import GasController

print("✓ Imports successful")
print(f"  Script dir    : {SCRIPT_DIR}")
print(f"  Parameters dir: {PARAMS_DIR}")
print(f"  Opt config dir: {OPT_CONFIG_DIR}")
print(f"  Config file   : {CONFIG_FILE_NAME}")


✓ Imports successful
  Script dir    : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_5_mode_comparison_adjustedexcesskinetics
  Parameters dir: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_5_mode_comparison_adjustedexcesskinetics\parameters
  Opt config dir: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_5_mode_comparison_adjustedexcesskinetics\optimization_config
  Config file   : campaign_config.yaml


In [2]:
# ==============================================================================
# 2. Load Campaign Config & Parameters
#
# ▶ SET SIM_ID MANUALLY to the simulation you want to troubleshoot.
#   Valid IDs are the "id" fields in campaign_config.yaml (e.g. "batch_E").
# ==============================================================================
SIM_ID = "cf_tbs"   # <── change this to the simulation you want to inspect
print(f"\n Simulation ID: {SIM_ID}")


 Simulation ID: cf_tbs


In [3]:
# ── Load campaign configuration ────────────────────────────────────────────────
config_path = CAMPAIGN_DIR / CONFIG_FILE_NAME
with open(config_path, encoding="utf-8") as f:
    campaign_config = yaml.safe_load(f)

# Resolve campaign defaults, mode defaults, and simulation overrides
sim_config = resolve_simulation_config(campaign_config, SIM_ID)
opt_settings = campaign_config["optimization_settings"]

# ── Simulation settings ────────────────────────────────────────────────────────
# Convert string names from YAML to enum objects using EnumClass["NAME"] lookup.
# Optional strategies (null in YAML → None in Python) are handled with a conditional.
MODE = OperatingMode[sim_config["mode"]]
KINETICS = Kinetics[sim_config["kinetics"]]
FEED_STRATEGY = FeedStrategy[sim_config["feed_strategy"]] if sim_config["feed_strategy"] else None
PERFUSION_STRATEGY = (
    PerfusionStrategy[sim_config["perfusion_strategy"]]
    if sim_config["perfusion_strategy"]
    else None
)
SOLVER_METHOD = sim_config.get("solver_method", "LSODA")

# ── Parameter file paths ───────────────────────────────────────────────────────
PARAM_FILES = [
    PARAMS_DIR / Path(entry).with_suffix(".yaml")
    for entry in sim_config["parameter_sets"]
]

# ── Optimization settings ──────────────────────────────────────────────────────
if sim_config["optimize"]:
    OBJECTIVES = opt_settings["objectives"]
    WEIGHTS = [opt_settings["objective_weights"][obj] for obj in OBJECTIVES]
    SEED = opt_settings["seed"]
    WORKERS = opt_settings["workers"]
    TIMEOUT = opt_settings.get("timeout", 5.0)
    OPT_CONFIG_FILE = OPT_CONFIG_DIR / f"{sim_config['opt_config']}_optimization.yaml"

# ── Export filename ────────────────────────────────────────────────────────────
EXPORT_FILENAME = opt_settings["filename"].replace("{simulation_id}", SIM_ID)

print(f"✓ Campaign config loaded: {SIM_ID} — {sim_config['label']}")
print(f"  Mode               : {MODE}")
print(f"  Kinetics           : {KINETICS}")
print(f"  Feed strategy      : {FEED_STRATEGY}")
print(f"  Perfusion strategy : {PERFUSION_STRATEGY}")
print(f"  Solver method      : {SOLVER_METHOD}")
print(f"  Optimize           : {sim_config['optimize']}")
if sim_config["optimize"]:
    print(f"  Opt config         : {OPT_CONFIG_FILE.name}")
    print(f"  Objectives         : {OBJECTIVES}")
    print(f"  Weights            : {WEIGHTS}")
    print(f"  Seed / Workers     : {SEED} / {WORKERS}")
    print(f"  Timeout            : {TIMEOUT}")
print(f"  Export file        : {EXPORT_FILENAME}")

# ── Load parameters ────────────────────────────────────────────────────────────
params = SimulationParameters.load_parameters([str(p) for p in PARAM_FILES])
print("\n✓ Parameters loaded successfully")
print("  Files:")
for p in PARAM_FILES:
    print(f"    {p.relative_to(PARAMS_DIR)}")


KeyError: 'objectives'

In [ ]:
# ==============================================================================
# 3. Baseline Simulation
# ==============================================================================
baseline_sim = CellCultureSimulation(
    parameters=params,
    kinetics=KINETICS,
    operating_mode=MODE,
    feed_strategy=FEED_STRATEGY,
    perfusion_strategy=PERFUSION_STRATEGY,
    solver_method=SOLVER_METHOD,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False,
)

baseline_result = baseline_sim.run_simulation()

print("\n✓ Baseline simulation completed")
print("\nBaseline Performance Metrics:")
for key, value in baseline_sim.data.summary_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")


run_simulation execution time: 1607.758 ms

✓ Baseline simulation completed

Baseline Performance Metrics:
  STY_kg_m3_day: 0.0890
  medium_use_efficiency_kg_m3: 0.4879
  protein_use_efficiency_kg_kg: 0.1512
  nutrient_use_efficiency_kg_kg: 0.0845
  nutrient_consumption_efficiency_kg_kg: 0.1121
  batch_time_range_h: 117.7852


## Mass Balance Check

In [ ]:
baseline_data = baseline_sim.data

print("=" * 70)
print("EXTERNAL BOUNDARY CHON CLOSURE (simulated window)")
print("=" * 70)
boundary_closure = baseline_data.compute_boundary_closure()

# Element-level summary
closure_df = baseline_data.build_closure_summary_df(boundary_closure)
print("\nElemental closure summary:")
display(closure_df)

# Max residual
print(f"\nMax absolute residual: {boundary_closure['max_abs_residual']:.6f} mol-atoms")

# Component basis
print("\n" + "-" * 70)
print("Component basis:")
print("-" * 70)
component_df = baseline_data.build_component_specs_df(boundary_closure['component_specs'])
display(component_df)



EXTERNAL BOUNDARY CHON CLOSURE (simulated window)

Elemental closure summary:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,2.826833e+03,2.826816e+03,-0.017117,-0.000606
1,H,1.343861e+06,1.343863e+06,1.854222,0.000138
2,O,6.727890e+05,6.727975e+05,8.449655,0.001256
3,N,3.268627e+02,3.268595e+02,-0.003262,-0.000998



Max absolute residual: 8.449655 mol-atoms

----------------------------------------------------------------------
Component basis:
----------------------------------------------------------------------


,Component,Side,Description,Amount,Amount Unit,C,H,O,N
0,Fresh glucose in,Input,Total glucose entering with fresh initial char...,2.511883e+02,mol,6.00000,12.000000,6.000000,0.000000
1,Fresh glutamine in,Input,Total glutamine entering with fresh initial ch...,3.823286e+01,mol,5.00000,10.000000,3.000000,2.000000
2,Fresh amino acids in,Input,Total lumped amino-acid pool entering with fre...,1.644179e+02,mol,4.73442,9.982912,1.516180,1.520349
3,Fresh lactate in,Input,Lactate entering from fresh external media only.,0.000000e+00,mol,3.00000,6.000000,3.000000,0.000000
4,Fresh ammonium in,Input,Ammonium entering from fresh external media only.,2.273737e-16,mol,0.00000,3.000000,0.000000,1.000000
5,O2 in,Input,Total oxygen crossing the reactor boundary int...,4.055045e+02,mol,0.00000,0.000000,2.000000,0.000000
6,CO2 in,Input,Total inorganic carbon entering from fresh med...,3.481389e+02,mol,1.00000,0.000000,2.000000,0.000000
7,"Medium, base, and acid water in",Input,Carrier water entering with fresh medium charg...,6.694100e+05,mol,0.00000,2.000000,1.000000,0.000000
8,DCM in,Input,"Dry biomass entering via inoculum, only with f...",1.977369e+00,Cmol,1.00000,1.641001,0.323461,0.214688
9,Glucose out,Output,Total glucose leaving in external liquid outle...,1.205963e-02,mol,6.00000,12.000000,6.000000,0.000000


In [ ]:
# ==============================================================================
# CHON Closure Delta Report — Boundary vs Stoichiometric
# ==============================================================================
# Compares the integrated kinetic-rate predictions (stoichiometric closure)
# against the actual reactor boundary fluxes to surface discrepancies.
print("\n" + "=" * 70)
print("CHON CLOSURE DELTA (boundary − stoichiometric)")
print("=" * 70)

try:
    delta_result = baseline_data.compute_closure_delta()

    # 1. Element-level comparison
    delta_summary_df = SimulationData.build_delta_summary_df(
        delta_result['stoich_closure'], delta_result['boundary_closure']
    )
    print("\nElement-level comparison:")
    display(delta_summary_df)

    # 2. Per-component delta
    print("\n" + "-" * 70)
    print("Per-component delta basis:")
    print("-" * 70)
    delta_component_df = SimulationData.build_delta_component_specs_df(delta_result['component_specs'])
    display(delta_component_df)

    print(f"\nMax absolute delta residual: {delta_result['max_abs_residual']:.6f} mol-atoms")

except ValueError as exc:
    print(f"\n⚠ Delta report unavailable: {exc}")
    print("  (Requires performance_mode=False for stoichiometric rate trajectories.)")


CHON CLOSURE DELTA (boundary − stoichiometric)

Element-level comparison:


,Element,Stoich Reactants (mol-atoms),Stoich Products (mol-atoms),Stoich Residual (mol-atoms),Boundary Reactants (mol-atoms),Boundary Products (mol-atoms),Boundary Residual (mol-atoms),Delta Residual (mol-atoms)
0,C,1763.917354,1763.907731,-0.009623,2.826833e+03,2.826816e+03,-0.017117,-0.007494
1,H,3538.009820,3537.987313,-0.022507,1.343861e+06,1.343863e+06,1.854222,1.876730
2,O,2453.235505,2453.229139,-0.006366,6.727890e+05,6.727975e+05,8.449655,8.456021
3,N,95.389352,95.386094,-0.003258,3.268627e+02,3.268595e+02,-0.003262,-0.000004



----------------------------------------------------------------------
Per-component delta basis:
----------------------------------------------------------------------


,Component,Side,Description,Stoich Amount,Boundary Net Amount,Delta Amount,Amount Unit,C,H,O,N
0,Glucose consumed delta,Input,Difference between boundary net flux and integ...,251.161012,251.176194,0.015182,mol,6.00000,12.000000,6.000000,0.000000
1,Glutamine consumed delta,Input,Difference between boundary net flux and integ...,32.648397,32.649439,0.001042,mol,5.00000,10.000000,3.000000,2.000000
2,Amino acids consumed delta,Input,Difference between boundary net flux and integ...,19.793193,19.795179,0.001986,mol,4.73442,9.982912,1.516180,1.520349
3,O2 consumed delta,Input,Difference between boundary net flux and integ...,409.157098,405.383869,-3.773229,mol,0.00000,0.000000,2.000000,0.000000
4,Dry biomass delta,Output,Difference between boundary net flux and integ...,266.064519,266.091182,0.026664,Cmol,1.00000,1.641001,0.323461,0.214688
5,Lactate produced delta,Output,Difference between boundary net flux and integ...,353.634762,353.662855,0.028093,mol,3.00000,6.000000,3.000000,0.000000
6,Ammonium produced delta,Output,Difference between boundary net flux and integ...,38.265188,38.264563,-0.000626,mol,0.00000,3.000000,0.000000,1.000000
7,CO2 produced delta,Output,Difference between boundary net flux and integ...,436.938926,436.926190,-0.012736,mol,1.00000,0.000000,2.000000,0.000000
8,H2O produced delta,Output,Difference between boundary net flux and integ...,432.385540,433.324898,0.939358,mol,0.00000,2.000000,1.000000,0.000000



Max absolute delta residual: 8.456021 mol-atoms


In [ ]:
baseline_sim.data.plot_kinetics()

In [ ]:
baseline_sim.data.plot_stoichiometric_yields()

In [ ]:
baseline_sim.data.plot_volume_feed(batch_number=1)

In [ ]:
fig_list = baseline_sim.data.plot_profiles()
for fig in fig_list:
    fig.show()

## Optimization

In [ ]:
# ==============================================================================
# 4. Optimization
#
# Multi-objective optimization using differential evolution. Objectives, weights,
# seed, and workers are all read from campaign_config.yaml → optimization_settings.
# The optimization parameter bounds come from the opt_config file resolved above.
# ==============================================================================
from animakinetica.analysis.optimization import OptimizationManager

# ── Campaign normalization scalars ─────────────────────────────────────────────
# These are the max objective values observed across all baseline simulations and are
# used as a consistent divisor so weights carry identical meaning for every simulation.
NORM_FILE_NAME = "campaign_normalization.yaml"
NORM_FILE = CAMPAIGN_DIR / NORM_FILE_NAME

if NORM_FILE.exists():
    with open(NORM_FILE) as f:
        campaign_norm = yaml.safe_load(f)
    print(f"✓ Loaded campaign normalization from {NORM_FILE.name}")
    for obj, val in campaign_norm.items():
        if isinstance(val, (int, float)):
            print(f"  {obj}: {val:.4g}")
else:
    campaign_norm = None
    print("  ⚠ campaign_normalization.yaml not found; run create_normalization_values.ipynb first.")
    print("    Falling back to per-simulation baseline normalization.")

# ── Initialize OptimizationManager ─────────────────────────────────────────────
opt_manager = OptimizationManager(
    simulation=baseline_sim,
    opt_config_file=str(OPT_CONFIG_FILE),
)

print(f"\n✓ OptimizationManager initialized")
print(f"  Config     : {OPT_CONFIG_FILE.name}")
print(f"  Parameters : {len(opt_manager.opt_config)}")
print("\nParameters to optimize:")
for param, cfg in opt_manager.opt_config.items():
    print(f"  {param}: bounds={cfg['bounds']}")

✓ Loaded campaign normalization from campaign_normalization.yaml
  STY_kg_m3_day: 0.31
  nutrient_use_efficiency_kg_kg: 0.04284
Searching specified optimization config: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_5_mode_comparison_adjustedexcesskinetics\optimization_config\BSF_optimization.yaml
Loading optimization config from: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_5_mode_comparison_adjustedexcesskinetics\optimization_config\BSF_optimization.yaml
Parsing...
Parsed and loaded configuration containing 5 optimization parameters.

✓ OptimizationManager initialized
  Config     : BSF_optimization.yaml
  Parameters : 5

Parameters to optimize:
  initial_conditions.C_Glc_init: bounds=(0.01, 100)
  initial_conditions.C_Gln_init: bounds=(0.01, 100)
  initial_conditions.C_AA_init: bounds=(0.01, 100)
  medium_properties.osm_baseline: bounds=(200, 300)
  fed_batch.C_Glc_batch_trigger: bounds=(0.01, 10)


In [ ]:
# # ── Run optimization ───────────────────────────────────────────────────────────
# opt_manager.optimize_parameters(
#     objectives=OBJECTIVES,
#     weights=WEIGHTS,
#     workers=WORKERS,
#     seed=SEED,
#     timeout=TIMEOUT,
#     verbose=True,
#     normalization_values=campaign_norm,
# )

# print("\n✓ Optimization completed")
# print("\nOptimal parameters found:")
# for param_name, value in opt_manager.optimal_parameters.items():
#     cfg = opt_manager.opt_config[param_name]
#     print(f"  {param_name}: {value:.6g} {cfg['unit']}")

In [ ]:
# ==============================================================================
# 5. Performance Comparison
# ==============================================================================
comparison_df = opt_manager.compare_performance(verbose=True)
comparison_df

ValueError: No optimal parameters available. Run optimize_parameters() first.

In [ ]:
# ==============================================================================
# 6. Final Simulation Check
# ==============================================================================
final_sim = opt_manager.optimized_simulation

print("\nFinal Performance Metrics:")
for key, value in final_sim.data.summary_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")


Final Performance Metrics:
  STY_kg_m3_day: 0.0715
  medium_use_efficiency_kg_m3: 0.4767
  protein_use_efficiency_kg_kg: 0.0255
  nutrient_use_efficiency_kg_kg: 0.0247
  nutrient_consumption_efficiency_kg_kg: 0.1080
  batch_time_range_h: 23.0145


In [ ]:
# ── Optimized simulation visualizations ────────────────────────────────────────
final_sim.data.plot_kinetics()

In [ ]:
final_sim.data.plot_stoichiometric_yields()

In [ ]:
final_sim.data.plot_volume_feed()

In [ ]:
fig_list = final_sim.data.plot_profiles()
for fig in fig_list:
    fig.show()

In [ ]:
# ==============================================================================
# 7. Export Results
#
# Three outputs saved to the local export/ directory:
#   c4_{SIM_ID}_opt_{timestamp}.xlsx         — full Excel export
#   c4_{SIM_ID}_opt_{timestamp}.pkl          — complete simulation state (pickle)
#   c4_{SIM_ID}_opt_{timestamp}.parameters.yaml — optimized parameter set
# ==============================================================================
EXPORT_DIR = SCRIPT_DIR / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
# final_sim
baseline_sim.save(
    filename=EXPORT_FILENAME,
    file_format="excel",
    include_metadata=True,
    include_summary_stats=True,
    include_cultivation_summary=True,
    include_batch_data=True,
    include_parameters=False,
    include_pickle=True,
    # include_opt_parameters=True,
    overwrite=True,
    export_dir=EXPORT_DIR,
)

print(f"\n✓ Export complete")
print(f"  Directory : {EXPORT_DIR}")
print("  Files written:")
for f in sorted(EXPORT_DIR.rglob("*")):
    if f.is_file():
        print(f"    {f.relative_to(EXPORT_DIR)}  ({f.stat().st_size / 1024:.1f} KB)")

Simulation data exported to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export\c4_fedbatch_constant_opt_20260701_1317\c4_fedbatch_constant_opt_20260701_1317.xlsx
CellCultureSimulation pickle saved to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export\c4_fedbatch_constant_opt_20260701_1317\c4_fedbatch_constant_opt_20260701_1317.pkl

✓ Export complete
  Directory : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export
  Files written:
    c4_fedbatch_constant_opt_20260629_1522\c4_fedbatch_constant_opt_20260629_1522.pkl  (1726.9 KB)
    c4_fedbatch_constant_opt_20260629_1522\c4_fedbatch_constant_opt_20260629_1522.xlsx  (640.1 KB)
    c4_fedbatch_constant_opt_20260629_1522\c4_fedbatch_constant_opt_20260629_1522_opt_params.yaml  (1.1 KB)
